Common Tools Integration
Pydantic AI includes a native "Common Tools" library for standard tasks like Web Searching. Instead of re-inventing the wheel, we can simply import pre-built toolsets into our Agent.

In this example, we will attach the native tavily_search_tool without having to write our own Python function.

Importing the Pre-built Tool
Pydantic AI makes it incredibly simple. You do not even need an @agent.tool decorator! Instead, you pass the tavily_search_tool directly into the Agent's tools list when initializing it.

In [6]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from openai import OpenAI

load_dotenv()

# -----------------------
# API KEYS
# -----------------------
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("Missing TAVILY_API_KEY")

# -----------------------
# Clients
# -----------------------
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

tavily = TavilyClient(api_key=TAVILY_API_KEY)

# -----------------------
# Tool
# -----------------------
def web_search(query: str) -> str:
    print(f"\n[Search] {query}")

    res = tavily.search(query=query, search_depth="basic")

    return "\n".join(
        f"- {r['title']}: {r['content']}"
        for r in res.get("results", [])
    )

# -----------------------
# Agent loop (simple + reliable)
# -----------------------
question = "What year was the James Webb Space Telescope launched?"

search_result = web_search(question)

messages = [
    {
        "role": "system",
        "content": "You are a precise research assistant. Use provided search results."
    },
    {
        "role": "user",
        "content": f"Question: {question}\n\nSearch results:\n{search_result}"
    }
]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    temperature=0.2
)

print("\n--- FINAL ANSWER ---")
print(response.choices[0].message.content)


[Search] What year was the James Webb Space Telescope launched?

--- FINAL ANSWER ---
The James Webb Space Telescope was launched on December 25, 2021.


Execution
The agent will natively comprehend the tavily_search_tool capabilities and utilize them to pull the latest information from the web.

In [8]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from openai import OpenAI

load_dotenv()

# -----------------------
# Clients
# -----------------------
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# -----------------------
# TOOL (manual safe call)
# -----------------------
def search_web(query: str) -> str:
    print(f"[Web Search] {query}")

    res = tavily.search(query=query, search_depth="basic")

    return "\n".join(
        f"- {r['title']}: {r['content']}"
        for r in res.get("results", [])
    )

# -----------------------
# STEP 1: search first (NO model tool calls)
# -----------------------
question = "What are the top news headlines globally today?"

search_results = search_web(question)

# -----------------------
# STEP 2: send to LLM
# -----------------------
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "You are a news assistant. Use only provided search results."
        },
        {
            "role": "user",
            "content": f"""
Question: {question}

Search results:
{search_results}

Summarize the top global headlines clearly.
"""
        }
    ],
    temperature=0.2
)

print("\n--- FINAL ANSWER ---")
print(response.choices[0].message.content)

[Web Search] What are the top news headlines globally today?

--- FINAL ANSWER ---
Here are the top global headlines today:

1. **Ebola Outbreak**: The World Health Organization (WHO) has declared the Ebola outbreak in Congo and Uganda a global health emergency.
2. **Middle East Tensions**: A drone strike has sparked a fire at a UAE power plant, potentially threatening the Iran truce.
3. **Pakistan's Role in Ending War**: Pakistan has handed the US a revised Iranian proposal for ending the war, and has also deployed a jet squadron and thousands of troops to Saudi Arabia.
4. **Global Health and Conflict**: Ongoing conflicts in Iraq and Afghanistan, as well as international business news about the European and Asian economic markets, are also making headlines.

These are the top global news stories currently trending around the world.
